In [1]:
import mlflow
import os
import json
from dotenv import load_dotenv

from mlflow.genai import scorer


load_dotenv()

os.environ["MLFLOW_TRACKING_URI"] = "http://localhost:5000"
os.environ["MLFLOW_EXPERIMENT_NAME"] = "log_model_examples"

In [2]:
with mlflow.start_run(run_name="langchain_model"):
    model_info = mlflow.pyfunc.log_model(
        name="lc_model",
        python_model="lc_model.py",
    )

d:\youtube\TheAIGuy\NLP\mlflow_examples\.venv\Lib\site-packages\mlflow\pyfunc\__init__.py:3215: UserWarning: Failed to infer signature from type hint: Type hints must be wrapped in list[...] because MLflow assumes the predict method to take multiple input instances. Specify your type hint as `list[dict]` for a valid signature.
  signature_from_type_hints = _infer_signature_from_type_hints(
2025/10/24 23:54:57 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
2025/10/24 23:54:57 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


🏃 View run langchain_model at: http://localhost:5000/#/experiments/706284548796550911/runs/a0aa51c5687049279d27c1d0346a4b5e
🧪 View experiment at: http://localhost:5000/#/experiments/706284548796550911


In [3]:
model_info.model_uri

'models:/m-7836000289fc47a4b9a7a79a346ed31e'

In [4]:
# # Load AG News dataset from Hugging Face as pandas dataframe
# from datasets import load_dataset
# dataset = load_dataset("ag_news", split="train")
# df = dataset.to_pandas()

# df["label"] = df["label"].map({0: "World", 1: "Sports", 2: "Business", 3: "Science"})

# df = df.sample(frac=1).reset_index(drop=True)


# NUM_SAMPLES = 20
# train_data = []
# for i in range(NUM_SAMPLES):
#     article = df.iloc[i]["text"]
#     expected = df.iloc[i]["label"]
#     eval_dict = {
#         "inputs": {"article": article},
#         "expectations": {"expected_response": expected},
#     }
#     train_data.append(eval_dict)

# # Save train_data list to a jsonl file
# import json
# with open("train_data.jsonl", "w") as f:
#     json.dump(train_data, f)


In [5]:
with open("train_data.jsonl", "r") as f:
    train_data = json.load(f)

train_data = train_data[:10]

In [6]:
model = mlflow.pyfunc.load_model(model_uri=model_info.model_uri)

d:\youtube\TheAIGuy\NLP\mlflow_examples\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
from lc_model import LangchainModel

def predict_fn(article):
    response = LangchainModel.predict(article)
    return response


In [8]:
@scorer
def exact_match(outputs, expectations):
    expectations = expectations["expected_response"]
    return outputs == expectations


results = mlflow.genai.evaluate(
    data=train_data,
    scorers=[exact_match],
    predict_fn=predict_fn,
    model_id=model.model_id,
)

2025/10/24 23:55:02 INFO mlflow.genai.utils.data_validation: Testing model prediction with the first sample in the dataset.


content='Science' additional_kwargs={} response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'grounding_metadata': {}, 'model_provider': 'google_genai'} id='lc_run--15b9a48d-f96c-4b0a-b15d-d76f801bcba6-0' usage_metadata={'input_tokens': 86, 'output_tokens': 1, 'total_tokens': 87, 'input_token_details': {'cache_read': 0}}


2025/10/24 23:55:16 INFO mlflow.tracking.fluent: Active model is set to the logged model with ID: m-7836000289fc47a4b9a7a79a346ed31e
2025/10/24 23:55:16 INFO mlflow.tracking.fluent: Use `mlflow.set_active_model` to set the active model to a different one if needed.
Evaluating:   0%|          | 0/10 [Elapsed: 00:00, Remaining: ?] 

content='Business' additional_kwargs={} response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'grounding_metadata': {}, 'model_provider': 'google_genai'} id='lc_run--725a610a-c346-474e-bff6-036ca49467c2-0' usage_metadata={'input_tokens': 65, 'output_tokens': 1, 'total_tokens': 66, 'input_token_details': {'cache_read': 0}}


Evaluating:  20%|██        | 2/10 [Elapsed: 00:17, Remaining: 01:09] 

...

Evaluating:  80%|████████  | 8/10 [Elapsed: 01:21, Remaining: 00:20] 

content='Science' additional_kwargs={} response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'grounding_metadata': {}, 'model_provider': 'google_genai'} id='lc_run--5a2abda4-2ab5-4ef4-ac70-9badb2eca27d-0' usage_metadata={'input_tokens': 86, 'output_tokens': 1, 'total_tokens': 87, 'input_token_details': {'cache_read': 0}}


Evaluating:  90%|█████████ | 9/10 [Elapsed: 01:29, Remaining: 00:09] 

content='Business' additional_kwargs={} response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'grounding_metadata': {}, 'model_provider': 'google_genai'} id='lc_run--8ea8b2a4-521d-43b5-ba40-e1623f55529d-0' usage_metadata={'input_tokens': 104, 'output_tokens': 1, 'total_tokens': 105, 'input_token_details': {'cache_read': 0}}


Evaluating: 100%|██████████| 10/10 [Elapsed: 01:40, Remaining: 00:00] 
